# Scout de Jóvenes Promesas — Paso 2
## Feature Engineering, Modelo Predictivo y MLflow

**Input:** `outputs/paso1/scouting_dataset_v1.csv`
→ 31,405 jugadores × 48 variables · 17,035 con target disponible

**Objetivo de este paso:**
Construir un modelo que aprenda a predecir el **valor de mercado actual**
de un jugador joven a partir de sus características físicas, rendimiento
histórico, exposición internacional y trayectoria económica.

**Por qué predecir el valor de mercado actual y no el futuro (todavía):**
El valor actual es la etiqueta disponible. Una vez validado que el modelo
captura bien los patrones, el Paso 3 (Montecarlo) extrapola esos patrones
hacia escenarios futuros a 3 y 5 años.

**Pipeline de este notebook:**
1. Feature engineering (limpieza, imputación, nuevas variables)
2. Selección de features
3. Entrenamiento LightGBM + XGBoost con validación cruzada
4. Comparación de modelos
5. Análisis de residuos
6. Interpretabilidad con SHAP
7. Predicción sobre todos los jugadores
8. Registro de experimentos en MLflow

In [4]:
# Imports y configuración
# ===========================================================

import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance

import lightgbm as lgb
import xgboost as xgb
import shap
import mlflow
import mlflow.lightgbm
import mlflow.xgboost

warnings.filterwarnings("ignore")

DATA_PATH  = Path("../outputs/paso1/scouting_dataset_v1.csv")
OUTPUT_DIR = Path("../outputs/paso2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "primary":   "#1D9E75",
    "secondary": "#7F77DD",
    "accent":    "#EF9F27",
    "danger":    "#D85A30",
    "neutral":   "#888780",
    "light":     "#F4F3EE",
    "dark":      "#2C2C2A",
}
plt.rcParams.update({
    "figure.facecolor": PALETTE["light"],
    "axes.facecolor":   PALETTE["light"],
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size": 11,
})

def save_fig(fig, name):
    path = OUTPUT_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=PALETTE["light"])
    plt.close(fig)
    print(f"  ✓ {name}.png")

print("✓ Librerías cargadas")
print(f"  LightGBM:  {lgb.__version__}")
print(f"  XGBoost:   {xgb.__version__}")
print(f"  MLflow:    {mlflow.__version__}")

✓ Librerías cargadas
  LightGBM:  4.6.0
  XGBoost:   3.2.0
  MLflow:    3.11.1


In [5]:
# Carga del dataset
# ===========================================================

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Dataset cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
print(f"Con target (latest_value): {df_raw['latest_value'].notna().sum():,}")
df_raw.head(3)

Dataset cargado: 31,405 filas × 48 columnas
Con target (latest_value): 17,035


,player_id,player_name,date_of_birth,age,country_of_birth,citizenship,is_eu,position,main_position,foot,...,intl_matches,intl_goals,n_nat_teams,is_current_national,n_transfers,max_fee,total_fee,club_country,club_league,log_market_value
0,1000152,Sodick Adejumo (1000152),2004-01-08,21.0,France,France Nigeria,True,Midfield - Defensive Midfield,Midfield,right,...,0.0,0.0,0.0,0.0,5.0,0.0,0.0,NaN,NaN,NaN
1,1000153,Matheo Sánchez (1000153),2003-03-13,21.8,France,France,True,Attack - Left Winger,Attack,right,...,0.0,0.0,0.0,0.0,2.0,0.0,0.0,NaN,NaN,NaN
2,1000161,Lenny Stoltz (1000161),2000-06-13,24.6,France,France Tunisia,True,Attack - Centre-Forward,Attack,left,...,0.0,0.0,0.0,0.0,4.0,0.0,0.0,NaN,NaN,NaN


## 1. Feature Engineering

Antes de entrenar, el dataset crudo necesita varias transformaciones.
Los problemas identificados en el Paso 1 y cómo se resuelven:

| Problema identificado | Solución aplicada |
|---|---|
| `height == 0` en 7,969 jugadores | Reemplazar 0 por NaN, imputar con mediana por posición |
| `foot` con 23.3% nulos | Imputar con moda por posición |
| `country_of_birth` con 24.8% nulos | Derivar de `citizenship` cuando sea posible |
| `mins_per_goal` nulo en porteros y no goleadores | Centinela 9999 para porteros, mediana por posición para el resto |
| `value_growth_pct` con máximo de 3,599x | Cap en 50x (el 98% de los jugadores está por debajo) |
| Historial de valor con 29.1% nulos | Imputar con mediana global |
| 163 ligas distintas en `club_league` | Agrupar en 3 tiers: `tier1`, `tier2`, `other` |
| Multicolinealidad entre `value_max`, `value_mean`, `value_growth_abs` | Se conserva solo `value_max` y `value_growth_pct` |

**Features derivadas nuevas:**

- `intl_ratio`: partidos internacionales / temporadas activas → mide la
  constancia en la selección, no solo el total acumulado.
- `goals_per_season` / `assists_per_season` / `minutes_per_season`:
  normaliza el rendimiento por tiempo de carrera, evitando que jugadores
  veteranos parezcan mejores solo por acumulación.
- `has_minutes`: flag binario que distingue jugadores con minutos reales
  de los que están registrados pero sin actividad documentada (5,581 casos).
- `injury_rate`: lesiones / temporadas activas → mide propensión real,
  no acumulación absoluta.
- `was_elite`: flag si el valor máximo histórico superó 10M€ → señal
  de que el mercado ya reconoció el potencial del jugador.
- `league_tier`: nivel competitivo del club actual, agrupado en 3 categorías.

In [6]:
# Feature Engineering
# ===========================================================

def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transforma el dataset crudo en features listas para modelado.
    Retorna un DataFrame limpio con las columnas seleccionadas.
    """
    
    d = df.copy()

    # ── 1. Altura: reemplazar 0 por NaN (claramente mal codificados)
    d["height"] = d["height"].replace(0, np.nan)

    # ── 2. Imputación de altura por posición (mediana)
    height_medians = d.groupby("main_position")["height"].transform("median")
    d["height"] = d["height"].fillna(height_medians)

    # ── 3. Pie predominante: imputar con moda por posición
    foot_mode = d.groupby("main_position")["foot"].transform(
        lambda x: x.mode().iloc[0] if not x.mode().empty else "right"
    )
    d["foot"] = d["foot"].fillna(foot_mode)
    d["foot"] = d["foot"].fillna("right")  # fallback global

    # ── 4. País de nacimiento: imputar con ciudadanía cuando sea posible
    mask = d["country_of_birth"].isna() & d["citizenship"].notna()
    d.loc[mask, "country_of_birth"] = d.loc[mask, "citizenship"]

    # ── 5. Minutos por gol:
    #    - Porteros → no aplica, imputar con valor centinela alto
    #    - Otros sin goles → imputar con mediana por posición
    gk_mask = d["main_position"] == "Goalkeeper"
    d.loc[gk_mask, "mins_per_goal"] = 9999.0

    mpg_medians = d[~gk_mask].groupby("main_position")["mins_per_goal"].transform("median")
    d.loc[~gk_mask, "mins_per_goal"] = d.loc[~gk_mask, "mins_per_goal"].fillna(mpg_medians)

    # ── 6. Features del historial de valor: imputar con mediana global
    hist_cols = ["value_max","value_mean","value_growth_abs","value_growth_pct","n_valuations"]
    for c in hist_cols:
        if c in d.columns:
            d[c] = d[c].fillna(d[c].median())

    # ── 7. Cap de outliers extremos en value_growth_pct
    #    (3599x de crecimiento distorsiona el modelo)
    d["value_growth_pct"] = d["value_growth_pct"].clip(upper=50)

    # ── 8. Features derivadas nuevas

    # Ratio de participación internacional vs temporadas activas
    d["intl_ratio"] = (d["intl_matches"] / d["n_seasons_active"].replace(0, np.nan)).fillna(0).round(3)

    # Productividad por temporada
    d["goals_per_season"]   = (d["total_goals"]   / d["n_seasons_active"].replace(0, np.nan)).fillna(0).round(2)
    d["assists_per_season"] = (d["total_assists"]  / d["n_seasons_active"].replace(0, np.nan)).fillna(0).round(2)
    d["minutes_per_season"] = (d["total_minutes"]  / d["n_seasons_active"].replace(0, np.nan)).fillna(0).round(0)

    # Flag: jugador con minutos reales (no solo inscrito)
    d["has_minutes"] = (d["total_minutes"] > 0).astype(int)

    # Flag: jugador de élite (valor máximo histórico > 10M€)
    d["was_elite"] = (d["value_max"] > 10_000_000).astype(int)

    # Ratio lesiones vs temporadas (propensión a lesionarse)
    d["injury_rate"] = (d["n_injuries"] / d["n_seasons_active"].replace(0, np.nan)).fillna(0).round(3)

    # Liga agrupada por nivel (reduce 163 categorías a ~5)
    top_leagues = {
        "Premier League":   "tier1",
        "LaLiga":           "tier1",
        "Serie A":          "tier1",
        "Bundesliga":       "tier1",
        "Ligue 1":          "tier1",
        "Eredivisie":       "tier2",
        "Liga Portugal":    "tier2",
        "Jupiler Pro League": "tier2",
        "Championship":     "tier2",
        "2. Bundesliga":    "tier2",
        "Serie B":          "tier2",
        "LaLiga2":          "tier2",
        "Major League Soccer": "other",
    }
    d["league_tier"] = d["club_league"].map(top_leagues).fillna("other")

    # ── 9. Encoding de categóricas
    #    main_position y foot: label encoding (pocas categorías, ordinales aproximadas)
    le_pos  = LabelEncoder()
    le_foot = LabelEncoder()

    d["position_enc"] = le_pos.fit_transform(d["main_position"].astype(str))
    d["foot_enc"]     = le_foot.fit_transform(d["foot"].astype(str))

    # league_tier: label encoding
    le_tier = LabelEncoder()
    d["league_tier_enc"] = le_tier.fit_transform(d["league_tier"].astype(str))

    # is_eu: ya es bool → convertir a int
    d["is_eu"] = d["is_eu"].astype(int)

    return d, le_pos, le_foot, le_tier


df_eng, le_pos, le_foot, le_tier = feature_engineering(df_raw)
print(f"Dataset tras feature engineering: {df_eng.shape}")

Dataset tras feature engineering: (31405, 59)


## 2. Selección de features

Se seleccionaron **31 features** agrupadas en 7 bloques temáticos.

**Criterios de exclusión:**
- Columnas de identificación (`player_id`, `player_name`, `date_of_birth`)
- Columnas con información futura o target leakage (`latest_value`, `value_last`,
  `value_first` — ya incorporados en `value_max` y `value_growth_pct`)
- Columnas con multicolinealidad extrema (r > 0.95):
  - `value_mean` y `value_growth_abs` se excluyen por su correlación con `value_max`
  - `goal_contrib` se reemplaza por `goals_per_season` + `assists_per_season`
  - `n_competitions` se excluye por su correlación de 0.72 con `n_seasons_active`
- Columnas de texto libre (`position`, `citizenship`, `club_league` cruda,
  `club_country`) — reemplazadas por sus versiones codificadas

**Variable objetivo:** `log_market_value = log(1 + latest_value)`

La transformación logarítmica es necesaria porque:
- El valor de mercado sigue una distribución Pareto (muy sesgada a la derecha)
- Modela mejor el crecimiento porcentual que el absoluto
- Reduce el impacto de los valores extremos (Yamal 200M€, Haaland 180M€)
  sin descartarlos

In [7]:
# Selección de features y preparación del dataset
# ===========================================================

# Features seleccionadas para el modelo
# Se excluyen: IDs, nombres, fechas, columnas con alta multicolinealidad
# y el target crudo (se usa log_market_value)
FEATURES = [
    # Perfil
    "age", "height", "is_eu", "foot_enc", "position_enc",

    # Rendimiento carrera
    "total_goals", "total_assists", "total_minutes",
    "goal_contrib", "mins_per_goal",
    "n_seasons_active", "n_competitions",
    "total_yellow", "total_red",
    "goals_per_season", "assists_per_season", "minutes_per_season",
    "has_minutes",

    # Lesiones
    "n_injuries", "total_days_missed", "injury_rate",

    # Selección nacional
    "intl_matches", "intl_goals", "intl_ratio", "is_current_national",

    # Historial de valor
    "value_max", "value_growth_pct", "n_valuations",

    # Transferencias
    "n_transfers", "max_fee",

    # Contexto de club
    "league_tier_enc", "was_elite",
]

TARGET = "log_market_value"

# Filtrar solo jugadores con target disponible
df_model = df_eng[df_eng[TARGET].notna()][FEATURES + [TARGET, "player_name", "player_id"]].copy()

# Verificar nulos restantes
null_check = df_model[FEATURES].isnull().sum()
if null_check.sum() > 0:
    print("Nulos restantes en features:")
    print(null_check[null_check > 0])
    # Imputación de emergencia con mediana
    for col in FEATURES:
        if df_model[col].isnull().sum() > 0:
            df_model[col] = df_model[col].fillna(df_model[col].median())
else:
    print("Sin nulos en el dataset de modelado")

X = df_model[FEATURES].values
y = df_model[TARGET].values

print(f"\nDataset de modelado:")
print(f"  X: {X.shape}")
print(f"  y: {y.shape}")
print(f"  Target — media: {y.mean():.3f}, std: {y.std():.3f}")
print(f"  Target — min: {y.min():.2f}, max: {y.max():.2f}")

Sin nulos en el dataset de modelado

Dataset de modelado:
  X: (17035, 32)
  y: (17035,)
  Target — media: 12.693, std: 1.658
  Target — min: 9.21, max: 19.11


## 3. Estrategia de validación

Se usa **K-Fold con 5 particiones** (shuffle, random_state=42).

**Por qué K-Fold y no validación temporal:**
En un contexto ideal de series temporales se usaría validación temporal
(entrenar en temporadas anteriores, predecir en las siguientes).
Sin embargo, el target (`latest_value`) es una foto puntual del momento
actual, no una serie por temporada, por lo que K-Fold aleatorio es apropiado.

En el Paso 3 (Montecarlo) sí se incorpora la dimensión temporal
para proyectar valores futuros.

**Métricas de evaluación:**
- **RMSE** (escala log): penaliza errores grandes, útil para detectar
  si el modelo falla sistemáticamente en los extremos.
- **R²**: proporción de varianza explicada. Un R² de 0.80 significa que
  el modelo explica el 80% de las diferencias de valor entre jugadores.
- **MAE** (escala log y €): error promedio, más interpretable para
  comunicar resultados a un reclutador.

Las predicciones OOF (Out-Of-Fold) se usan para calcular las métricas
finales — cada jugador es predicho exactamente una vez por un modelo
que nunca lo vio en entrenamiento.

In [8]:
# Visualización del dataset de modelado
# ===========================================================

# Distribución del target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Target para el modelo: log(valor de mercado)", fontsize=13, fontweight="bold")

axes[0].hist(y, bins=60, color=PALETTE["primary"], edgecolor="white", lw=0.3)
axes[0].set_xlabel("log(1 + valor €)")
axes[0].set_ylabel("Jugadores")
axes[0].set_title("Distribución log-transformada")
axes[0].axvline(y.mean(), color=PALETTE["accent"], lw=2, ls="--", label=f"Media: {y.mean():.2f}")
axes[0].legend()

# Distribución reconvertida a M€
axes[1].hist(np.expm1(y) / 1e6, bins=70, color=PALETTE["secondary"], edgecolor="white", lw=0.3)
axes[1].set_xscale("log")
axes[1].set_xlabel("Valor de mercado (M€) — escala log")
axes[1].set_ylabel("Jugadores")
axes[1].set_title("Distribución reconvertida a € (verificación)")

fig.tight_layout()
save_fig(fig, "01_target_modelado")

# Features por posición
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Features clave por posición", fontsize=13, fontweight="bold")

for ax, (feat, label) in zip(axes.flatten(), [
    ("total_goals",   "Goles totales"),
    ("total_minutes", "Minutos totales"),
    ("intl_matches",  "Partidos internacionales"),
    ("n_injuries",    "Lesiones totales"),
]):
    for pos, color in zip(["Attack","Midfield","Defender","Goalkeeper"],
                           [PALETTE["danger"], PALETTE["secondary"], PALETTE["primary"], PALETTE["accent"]]):
        vals = df_model[df_model["position_enc"] == le_pos.transform([pos])[0]][feat] if pos in le_pos.classes_ else pd.Series()
        if len(vals) > 0:
            ax.hist(vals.clip(upper=vals.quantile(0.95)), bins=30,
                    alpha=0.5, color=color, label=pos, edgecolor="white", lw=0.2)
    ax.set_xlabel(label)
    ax.set_ylabel("Jugadores")
    ax.set_title(label)
    ax.legend(fontsize=8)

fig.tight_layout()
save_fig(fig, "02_features_por_posicion")

  ✓ 01_target_modelado.png
  ✓ 02_features_por_posicion.png


In [9]:
# Validación cruzada y entrenamiento LightGBM
# ===========================================================

# Configuración de MLflow
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("scout_promesas_paso2")

# Parámetros LightGBM
LGBM_PARAMS = {
    "objective":        "regression",
    "metric":           "rmse",
    "learning_rate":    0.05,
    "num_leaves":       63,
    "max_depth":        -1,
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "reg_alpha":        0.1,
    "reg_lambda":       0.1,
    "n_estimators":     500,
    "random_state":     42,
    "verbose":          -1,
}

# Validación cruzada K-Fold (5 folds)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds_lgbm = np.zeros(len(y))
models_lgbm    = []
feature_imps   = []

print("Entrenando LightGBM con 5-Fold CV...\n")

with mlflow.start_run(run_name="LightGBM_5fold"):

    mlflow.log_params(LGBM_PARAMS)
    mlflow.log_param("n_features",     len(FEATURES))
    mlflow.log_param("n_samples",      len(y))
    mlflow.log_param("cv_strategy",    "KFold-5")
    mlflow.log_param("target",         "log_market_value")

    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model = lgb.LGBMRegressor(**LGBM_PARAMS)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(period=-1)],
        )

        preds = model.predict(X_val)
        oof_preds_lgbm[val_idx] = preds

        fold_rmse = np.sqrt(mean_squared_error(y_val, preds))
        fold_r2   = r2_score(y_val, preds)
        print(f"  Fold {fold+1}:  RMSE={fold_rmse:.4f}  R²={fold_r2:.4f}  "
              f"best_iter={model.best_iteration_}")

        mlflow.log_metric(f"fold_{fold+1}_rmse", fold_rmse)
        mlflow.log_metric(f"fold_{fold+1}_r2",   fold_r2)

        models_lgbm.append(model)
        feature_imps.append(model.feature_importances_)

    # Métricas OOF globales
    oof_rmse_lgbm = np.sqrt(mean_squared_error(y, oof_preds_lgbm))
    oof_r2_lgbm   = r2_score(y, oof_preds_lgbm)
    oof_mae_lgbm  = mean_absolute_error(y, oof_preds_lgbm)

    mlflow.log_metric("oof_rmse", oof_rmse_lgbm)
    mlflow.log_metric("oof_r2",   oof_r2_lgbm)
    mlflow.log_metric("oof_mae",  oof_mae_lgbm)

    # Guardar el mejor modelo (fold con menor RMSE)
    best_fold = np.argmin([
        np.sqrt(mean_squared_error(y[val_idx], oof_preds_lgbm[val_idx]))
        for _, val_idx in kf.split(X, y)
    ])
    mlflow.lightgbm.log_model(models_lgbm[best_fold], "lgbm_model")

print(f"\n  ── LightGBM OOF Global ──")
print(f"  RMSE:  {oof_rmse_lgbm:.4f}")
print(f"  R²:    {oof_r2_lgbm:.4f}")
print(f"  MAE:   {oof_mae_lgbm:.4f}")

Entrenando LightGBM con 5-Fold CV...
  Fold 1:  RMSE=0.2926  R²=0.9682  best_iter=179
  Fold 2:  RMSE=0.2818  R²=0.9693  best_iter=216
  Fold 3:  RMSE=0.2862  R²=0.9709  best_iter=239
  Fold 4:  RMSE=0.2733  R²=0.9736  best_iter=333


2026/04/13 07:55:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Fold 5:  RMSE=0.2814  R²=0.9719  best_iter=170


2026/04/13 07:55:13 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  ── LightGBM OOF Global ──
  RMSE:  0.2831
  R²:    0.9708
  MAE:   0.1884


## Análisis — Resultados LightGBM

### Interpretación de métricas

El R² OOF indica qué tan bien el modelo generaliza a jugadores no vistos.
Para referencia:

| R² | Interpretación |
|---|---|
| > 0.85 | Excelente para datos de fútbol |
| 0.70 – 0.85 | Bueno, modelo útil para scouting |
| 0.50 – 0.70 | Moderado, útil pero con incertidumbre alta |
| < 0.50 | Insuficiente |

El RMSE en escala log se interpreta aproximadamente como el error
porcentual medio en la predicción del valor.

> **Nota sobre el MAE en €:** dado que el valor de mercado tiene una
> distribución muy sesgada (mediana 250K€, máximo 200M€), el MAE en €
> estará dominado por los errores en los jugadores más valiosos. Es
> más informativo segmentarlo por rango de valor, lo que se hace
> en el análisis de residuos.

## 4. XGBoost y comparacion con LightGBM

Se entrena un modeo XGBoost y se comparan los resultados en busca del mejor

In [10]:
# Entrenamiento XGBoost (comparación)
# ===========================================================

XGB_PARAMS = {
    "objective":        "reg:squarederror",
    "eval_metric":      "rmse",
    "learning_rate":    0.05,
    "max_depth":        6,
    "min_child_weight": 5,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "reg_alpha":        0.1,
    "reg_lambda":       0.5,
    "n_estimators":     500,
    "random_state":     42,
    "verbosity":        0,
    "early_stopping_rounds": 50,
}

oof_preds_xgb = np.zeros(len(y))
models_xgb    = []

print("Entrenando XGBoost con 5-Fold CV...\n")

with mlflow.start_run(run_name="XGBoost_5fold"):

    mlflow.log_params({k: v for k, v in XGB_PARAMS.items() if k != "early_stopping_rounds"})

    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model = xgb.XGBRegressor(**XGB_PARAMS)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        preds = model.predict(X_val)
        oof_preds_xgb[val_idx] = preds

        fold_rmse = np.sqrt(mean_squared_error(y_val, preds))
        fold_r2   = r2_score(y_val, preds)
        print(f"  Fold {fold+1}:  RMSE={fold_rmse:.4f}  R²={fold_r2:.4f}  "
              f"best_iter={model.best_iteration}")

        mlflow.log_metric(f"fold_{fold+1}_rmse", fold_rmse)
        mlflow.log_metric(f"fold_{fold+1}_r2",   fold_r2)
        models_xgb.append(model)

    oof_rmse_xgb = np.sqrt(mean_squared_error(y, oof_preds_xgb))
    oof_r2_xgb   = r2_score(y, oof_preds_xgb)
    oof_mae_xgb  = mean_absolute_error(y, oof_preds_xgb)

    mlflow.log_metric("oof_rmse", oof_rmse_xgb)
    mlflow.log_metric("oof_r2",   oof_r2_xgb)
    mlflow.log_metric("oof_mae",  oof_mae_xgb)
    mlflow.xgboost.log_model(models_xgb[0], "xgb_model")

print(f"\n  ── XGBoost OOF Global ──")
print(f"  RMSE:  {oof_rmse_xgb:.4f}")
print(f"  R²:    {oof_r2_xgb:.4f}")
print(f"  MAE:   {oof_mae_xgb:.4f}")

Entrenando XGBoost con 5-Fold CV...
  Fold 1:  RMSE=0.2893  R²=0.9690  best_iter=250
  Fold 2:  RMSE=0.2822  R²=0.9692  best_iter=278
  Fold 3:  RMSE=0.2831  R²=0.9715  best_iter=298
  Fold 4:  RMSE=0.2724  R²=0.9738  best_iter=449


2026/04/13 07:55:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Fold 5:  RMSE=0.2810  R²=0.9720  best_iter=177

  ── XGBoost OOF Global ──
  RMSE:  0.2817
  R²:    0.9711
  MAE:   0.1865


In [11]:
# Comparación de modelos
# ===========================================================

resultados = pd.DataFrame({
    "Modelo":   ["LightGBM", "XGBoost"],
    "OOF RMSE": [oof_rmse_lgbm, oof_rmse_xgb],
    "OOF R²":   [oof_r2_lgbm,   oof_r2_xgb],
    "OOF MAE":  [oof_mae_lgbm,  oof_mae_xgb],
})
print(resultados.to_string(index=False))

# Seleccionar el mejor modelo
mejor_modelo_nombre = "LightGBM" if oof_rmse_lgbm <= oof_rmse_xgb else "XGBoost"
mejor_modelo        = models_lgbm[0] if oof_rmse_lgbm <= oof_rmse_xgb else models_xgb[0]
oof_preds_mejor     = oof_preds_lgbm if oof_rmse_lgbm <= oof_rmse_xgb else oof_preds_xgb

print(f"\n  Mejor modelo: {mejor_modelo_nombre}")

# Plot comparación
fig, ax = plt.subplots(figsize=(9, 5))
x_pos = np.arange(3)
w = 0.35
metricas = ["OOF RMSE", "OOF R²", "OOF MAE"]
lgbm_vals = [oof_rmse_lgbm, oof_r2_lgbm, oof_mae_lgbm]
xgb_vals  = [oof_rmse_xgb,  oof_r2_xgb,  oof_mae_xgb]

ax.bar(x_pos - w/2, lgbm_vals, w, label="LightGBM", color=PALETTE["primary"])
ax.bar(x_pos + w/2, xgb_vals,  w, label="XGBoost",  color=PALETTE["secondary"])
ax.set_xticks(x_pos)
ax.set_xticklabels(metricas)
ax.set_title("Comparación LightGBM vs XGBoost — métricas OOF", fontweight="bold")
ax.legend()
for i, (lv, xv) in enumerate(zip(lgbm_vals, xgb_vals)):
    ax.text(i - w/2, lv + 0.003, f"{lv:.3f}", ha="center", fontsize=9)
    ax.text(i + w/2, xv + 0.003, f"{xv:.3f}", ha="center", fontsize=9)
fig.tight_layout()
save_fig(fig, "03_comparacion_modelos")

  Modelo  OOF RMSE   OOF R²  OOF MAE
LightGBM  0.283127 0.970846 0.188447
 XGBoost  0.281667 0.971145 0.186502

  Mejor modelo: XGBoost
  ✓ 03_comparacion_modelos.png


## 5. Análisis de residuos

Los residuos (valor real − valor predicho) revelan dónde falla el modelo
y si hay patrones sistemáticos.

**Qué buscar en los plots:**

- **Real vs Predicho:** los puntos deben distribuirse cerca de la línea
  diagonal. Desviaciones en los extremos indican que el modelo subestima
  a los jugadores más valiosos (regresión a la media) o sobreestima
  a los de menor valor.

- **Residuos vs Predicho:** si hay un patrón en forma de abanico
  (mayor dispersión para valores altos), indica heterocedasticidad —
  el modelo es menos preciso para jugadores de élite, lo cual es esperable
  dado que son casos atípicos con pocos ejemplos similares.

- **Distribución de residuos:** idealmente centrada en cero y
  aproximadamente normal. Un sesgo positivo o negativo indica que
  el modelo tiene un error sistemático en una dirección.

### Qué esperamos ver con esta data

- El modelo probablemente **subestimará** a jugadores como Lamine Yamal
  (17 años, 200M€) — su valor está impulsado por expectativas futuras
  que no están completamente capturadas en los stats actuales.
- Los residuos más grandes aparecerán en el segmento > 50M€ 
  (solo ~300 jugadores en el dataset).
- Para el 80% de los jugadores (valor entre 100K€ y 5M€), el modelo
  debería funcionar con buena precisión.

In [12]:
# Análisis de residuos y predicciones
# ===========================================================

# Predicciones OOF reconvertidas a €
y_real_eur  = np.expm1(y)
y_pred_eur  = np.expm1(oof_preds_mejor)
residuos    = y - oof_preds_mejor
residuos_eur = y_real_eur - y_pred_eur

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle(f"Análisis de residuos — {mejor_modelo_nombre}", fontsize=13, fontweight="bold")

# Real vs predicho (escala log)
axes[0, 0].scatter(y, oof_preds_mejor, alpha=0.2, s=8, color=PALETTE["primary"])
mn, mx = y.min(), y.max()
axes[0, 0].plot([mn, mx], [mn, mx], color=PALETTE["danger"], lw=1.5, ls="--", label="Predicción perfecta")
axes[0, 0].set_xlabel("log(valor real)")
axes[0, 0].set_ylabel("log(valor predicho)")
axes[0, 0].set_title("Real vs Predicho (escala log)")
axes[0, 0].legend(fontsize=9)

# Residuos vs predicho
axes[0, 1].scatter(oof_preds_mejor, residuos, alpha=0.2, s=8, color=PALETTE["secondary"])
axes[0, 1].axhline(0, color=PALETTE["danger"], lw=1.5, ls="--")
axes[0, 1].set_xlabel("log(valor predicho)")
axes[0, 1].set_ylabel("Residuo")
axes[0, 1].set_title("Residuos vs Predicho")

# Distribución de residuos
axes[1, 0].hist(residuos, bins=60, color=PALETTE["accent"], edgecolor="white", lw=0.3)
axes[1, 0].axvline(0, color=PALETTE["danger"], lw=1.5, ls="--")
axes[1, 0].set_xlabel("Residuo (log)")
axes[1, 0].set_ylabel("Frecuencia")
axes[1, 0].set_title("Distribución de residuos")

# Error en €: real vs predicho (top rango)
mask_top = y_real_eur < np.percentile(y_real_eur, 95)
axes[1, 1].scatter(y_real_eur[mask_top] / 1e6, y_pred_eur[mask_top] / 1e6,
                   alpha=0.2, s=8, color=PALETTE["primary"])
lim = max(y_real_eur[mask_top].max(), y_pred_eur[mask_top].max()) / 1e6
axes[1, 1].plot([0, lim], [0, lim], color=PALETTE["danger"], lw=1.5, ls="--")
axes[1, 1].set_xlabel("Valor real (M€)")
axes[1, 1].set_ylabel("Valor predicho (M€)")
axes[1, 1].set_title("Real vs Predicho en M€ (hasta P95)")

fig.tight_layout()
save_fig(fig, "04_residuos")

# Métricas en escala original (€)
rmse_eur = np.sqrt(mean_squared_error(y_real_eur, y_pred_eur))
mae_eur  = mean_absolute_error(y_real_eur, y_pred_eur)
print(f"\n  Métricas en escala original (€):")
print(f"  RMSE: {rmse_eur/1e6:.2f}M€")
print(f"  MAE:  {mae_eur/1e6:.2f}M€")

  ✓ 04_residuos.png

  Métricas en escala original (€):
  RMSE: 2.12M€
  MAE:  0.41M€


## 6. Interpretabilidad con SHAP

SHAP (SHapley Additive exPlanations) asigna a cada feature una contribución
individual a cada predicción específica, a diferencia de la importancia
nativa del modelo que es global y menos informativa.

**Cómo leer el SHAP summary plot (dot plot):**
- Cada punto es un jugador
- El eje X indica cuánto contribuye esa feature a la predicción
  (positivo = sube el valor predicho, negativo = lo baja)
- El color indica el valor de la feature:
  **rojo = valor alto**, **azul = valor bajo**

**Lo que esperamos ver:**

- `value_max`: puntos rojos a la derecha → jugadores con historial
  de valoraciones altas reciben predicciones más altas. Tiene sentido:
  el mercado tiene memoria.
- `intl_matches`: rojo a la derecha → más partidos internacionales,
  mayor valor predicho. La selección nacional es un validador externo
  del talento.
- `age`: azul a la derecha en algunos rangos → jugadores más jóvenes
  reciben un "bonus de potencial" en la predicción.
- `max_fee`: rojo a la derecha → el fee más alto pagado por un
  jugador es una señal muy potente del valor que el mercado le asigna.

> **Para el reclutador:** SHAP permite explicar por qué el modelo
> recomienda a un jugador específico, no solo qué posición ocupa
> en el ranking. Esto es esencial para justificar decisiones de fichaje.

In [13]:
# Importancia de features (modelo nativo)
# ===========================================================

# Importancia media de los 5 folds (LightGBM)
imp_matrix = np.array(feature_imps) if oof_rmse_lgbm <= oof_rmse_xgb else None

if imp_matrix is not None:
    imp_mean = imp_matrix.mean(axis=0)
    imp_std  = imp_matrix.std(axis=0)
else:
    imp_mean = models_xgb[0].feature_importances_
    imp_std  = np.zeros_like(imp_mean)

imp_df = pd.DataFrame({
    "feature":    FEATURES,
    "importance": imp_mean,
    "std":        imp_std,
}).sort_values("importance", ascending=False)

print("Top 15 features por importancia:")
print(imp_df.head(15).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 9))
top_n = imp_df.head(20)
ax.barh(top_n["feature"][::-1], top_n["importance"][::-1],
        xerr=top_n["std"][::-1], color=PALETTE["primary"],
        height=0.65, error_kw={"elinewidth": 1, "ecolor": PALETTE["neutral"]})
ax.set_xlabel("Importancia (media 5 folds)")
ax.set_title(f"Top 20 features más importantes — {mejor_modelo_nombre}", fontweight="bold")
fig.tight_layout()
save_fig(fig, "05_feature_importance")

Top 15 features por importancia:
           feature  importance  std
         was_elite    0.408223  0.0
         value_max    0.394431  0.0
  value_growth_pct    0.060404  0.0
           max_fee    0.049330  0.0
   league_tier_enc    0.028204  0.0
     total_assists    0.006409  0.0
      n_valuations    0.005951  0.0
        intl_ratio    0.004152  0.0
assists_per_season    0.003752  0.0
      intl_matches    0.003327  0.0
      goal_contrib    0.002850  0.0
       n_transfers    0.002526  0.0
      total_yellow    0.002385  0.0
             is_eu    0.002043  0.0
 total_days_missed    0.001953  0.0
  ✓ 05_feature_importance.png


In [14]:
# Interpretabilidad con SHAP
# ===========================================================

print("Calculando SHAP values...")

# Usar el modelo del primer fold sobre una muestra representativa
X_df = pd.DataFrame(X, columns=FEATURES)
sample_idx = np.random.RandomState(42).choice(len(X), size=min(2000, len(X)), replace=False)
X_sample = X_df.iloc[sample_idx]

if oof_rmse_lgbm <= oof_rmse_xgb:
    explainer   = shap.TreeExplainer(models_lgbm[0])
else:
    explainer   = shap.TreeExplainer(models_xgb[0])

shap_values = explainer.shap_values(X_sample)

# SHAP summary plot
fig, ax = plt.subplots(figsize=(11, 9))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURES,
                  show=False, plot_type="dot", max_display=20)
plt.title(f"SHAP Values — impacto de cada feature en la predicción\n({mejor_modelo_nombre})",
          fontweight="bold", pad=12)
save_fig(plt.gcf(), "06_shap_summary")

# SHAP bar (importancia media absoluta)
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURES,
                  show=False, plot_type="bar", max_display=20)
plt.title("SHAP — importancia media absoluta (top 20)", fontweight="bold", pad=12)
save_fig(plt.gcf(), "07_shap_bar")

print("  ✓ SHAP values calculados")

Calculando SHAP values...
  ✓ 06_shap_summary.png
  ✓ 07_shap_bar.png
  ✓ SHAP values calculados


## 7. Predicciones sobre todos los jugadores

El modelo final se reentrena con **todos los datos disponibles** (sin
holdout) para maximizar la información aprendida antes de predecir.

Se aplica sobre los 31,405 jugadores del dataset, incluyendo los 14,370
que no tenían valor de mercado registrado. Para estos últimos, la
predicción es especialmente valiosa porque identifica jugadores
que el mercado aún no ha valorizado formalmente pero que el modelo
detecta como prometedores por sus estadísticas.

**El ranking resultante tiene dos usos:**
1. Validar que el modelo recupera a los jugadores conocidos en las
   primeras posiciones (sanity check)
2. Descubrir jugadores con alto valor predicho pero bajo o nulo valor
   de mercado registrado → candidatos reales de scouting

In [16]:
# Predicciones sobre todos los jugadores
# ===========================================================

# Reentrenar con todos los datos para predicciones finales
print("Reentrenando modelo final con todos los datos...")

if oof_rmse_lgbm <= oof_rmse_xgb:
    modelo_final = lgb.LGBMRegressor(**{**LGBM_PARAMS, "n_estimators": 400})
else:
    modelo_final = xgb.XGBRegressor(**{k: v for k, v in XGB_PARAMS.items()
                                        if k != "early_stopping_rounds"})

modelo_final.fit(X, y)

# Aplicar sobre todo el dataset (con y sin target)
df_all = df_eng.copy()
feats_check = [c for c in FEATURES if c in df_all.columns]

# Imputación de emergencia para filas sin target
for col in feats_check:
    if df_all[col].isnull().sum() > 0:
        df_all[col] = df_all[col].fillna(df_all[col].median())

X_all  = df_all[feats_check].values
y_pred_all = modelo_final.predict(X_all)
df_all["pred_log_value"]   = y_pred_all
df_all["pred_market_value_eur"] = np.expm1(y_pred_all)

# Ranking de promesas con mayor valor predicho
cols_out = [
    "player_name", "age", "main_position", "current_club_name",
    "club_country", "latest_value", "pred_market_value_eur",
    "total_goals", "total_assists", "intl_matches", "n_injuries",
]
cols_out = [c for c in cols_out if c in df_all.columns]

ranking = (
    df_all[cols_out]
    .sort_values("pred_market_value_eur", ascending=False)
    .reset_index(drop=True)
)
ranking.index += 1

print("\n  TOP 30 PROMESAS — por valor de mercado PREDICHO:")
print(ranking.head(50).to_string())

# Guardar ranking completo
ranking_path = OUTPUT_DIR / "ranking_promesas.csv"
ranking.to_csv(ranking_path, index=True)
print(f"\n  ✓ Ranking guardado: {ranking_path}")

# Guardar dataset completo con predicciones
pred_path = OUTPUT_DIR / "scouting_dataset_v2_con_predicciones.csv"
df_all.to_csv(pred_path, index=False)
print(f"  ✓ Dataset con predicciones: {pred_path}")

Reentrenando modelo final con todos los datos...

  TOP 30 PROMESAS — por valor de mercado PREDICHO:
                       player_name   age main_position    current_club_name club_country  latest_value  pred_market_value_eur  total_goals  total_assists  intl_matches  n_injuries
1            Lamine Yamal (937958)  17.5        Attack         FC Barcelona        Spain   200000000.0            179975840.0         27.0           37.0          44.0         6.0
2          Erling Haaland (418560)  24.4        Attack      Manchester City      England   180000000.0            176015744.0        262.0           57.0          98.0        15.0
3         Vinicius Junior (371998)  24.5        Attack          Real Madrid        Spain   170000000.0            168025568.0        126.0           93.0          70.0         9.0
4         Jude Bellingham (581678)  21.5      Midfield          Real Madrid        Spain   180000000.0            167175232.0         66.0           55.0           0.0         7.0

In [17]:
# Visualización del ranking final
# ===========================================================

top30 = ranking.head(30).copy()

fig, ax = plt.subplots(figsize=(13, 10))
colors_pos = {
    "Attack":     PALETTE["danger"],
    "Midfield":   PALETTE["secondary"],
    "Defender":   PALETTE["primary"],
    "Goalkeeper": PALETTE["accent"],
}
bar_colors = [colors_pos.get(p, PALETTE["neutral"]) for p in top30["main_position"]]

ax.barh(range(len(top30)), top30["pred_market_value_eur"] / 1e6,
        color=bar_colors, height=0.72, alpha=0.85)

# Superponer valor real si existe
for i, (_, row) in enumerate(top30.iterrows()):
    if pd.notna(row.get("latest_value")) and row["latest_value"] > 0:
        ax.plot(row["latest_value"] / 1e6, i, "o",
                color=PALETTE["dark"], ms=5, zorder=5)

ax.set_yticks(range(len(top30)))
ax.set_yticklabels([
    f"{row['player_name'].split('(')[0].strip()} · {int(row['age'])}a · {row['main_position']}"
    for _, row in top30.iterrows()
], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Valor de mercado predicho (M€)")
ax.set_title("Top 30 jóvenes promesas — ranking por valor predicho\n(● = valor real de mercado)",
             fontweight="bold", pad=12)

from matplotlib.patches import Patch
legend_el = [Patch(color=c, label=l) for l, c in colors_pos.items()]
legend_el.append(plt.Line2D([0], [0], marker="o", color=PALETTE["dark"],
                              label="Valor real", linewidth=0, markersize=6))
ax.legend(handles=legend_el, loc="lower right", fontsize=9)

fig.tight_layout()
save_fig(fig, "08_ranking_final")

  ✓ 08_ranking_final.png


## 8. MLflow — Registro de experimentos

MLflow registra automáticamente cada experimento con sus parámetros,
métricas y artefactos. Para visualizar los resultados:

```bash
# Desde la carpeta raíz del proyecto
mlflow ui
# Abrir en el navegador: http://localhost:5000
```

Se registran 3 runs:
- `LightGBM_5fold`: métricas por fold + OOF global
- `XGBoost_5fold`: métricas por fold + OOF global
- `modelo_final_produccion`: modelo ganador, métricas finales y artefactos

Esto permite en el futuro comparar versiones del modelo cuando se
incorporen nuevas features o datos de nuevas temporadas.

In [18]:
# Resumen y logging final en MLflow
# ===========================================================

with mlflow.start_run(run_name="modelo_final_produccion"):
    mlflow.log_param("modelo",         mejor_modelo_nombre)
    mlflow.log_param("n_features",     len(FEATURES))
    mlflow.log_param("n_train",        len(y))
    mlflow.log_param("target",         "log_market_value")
    mlflow.log_metric("oof_rmse",      oof_rmse_lgbm if oof_rmse_lgbm <= oof_rmse_xgb else oof_rmse_xgb)
    mlflow.log_metric("oof_r2",        oof_r2_lgbm   if oof_rmse_lgbm <= oof_rmse_xgb else oof_r2_xgb)
    mlflow.log_metric("oof_mae",       oof_mae_lgbm  if oof_rmse_lgbm <= oof_rmse_xgb else oof_mae_xgb)
    mlflow.log_metric("rmse_eur_M",    rmse_eur / 1e6)
    mlflow.log_metric("mae_eur_M",     mae_eur / 1e6)
    mlflow.log_artifact(str(ranking_path))
    mlflow.log_artifact(str(pred_path))

print("\n" + "═"*62)
print("  RESUMEN PASO 2")
print("═"*62)
print(f"""
  Modelo seleccionado:      {mejor_modelo_nombre}
  Features utilizadas:      {len(FEATURES)}
  Jugadores entrenados:     {len(y):,}

  Métricas OOF (escala log):
    RMSE:   {(oof_rmse_lgbm if oof_rmse_lgbm <= oof_rmse_xgb else oof_rmse_xgb):.4f}
    R²:     {(oof_r2_lgbm   if oof_rmse_lgbm <= oof_rmse_xgb else oof_r2_xgb):.4f}
    MAE:    {(oof_mae_lgbm  if oof_rmse_lgbm <= oof_rmse_xgb else oof_mae_xgb):.4f}

  Métricas en escala original:
    RMSE:   {rmse_eur/1e6:.2f}M€
    MAE:    {mae_eur/1e6:.2f}M€

  Outputs generados:
    ranking_promesas.csv
    scouting_dataset_v2_con_predicciones.csv
    08 plots PNG en outputs/paso2/
    Experimentos en mlruns/ (ver con: mlflow ui)
""")


══════════════════════════════════════════════════════════════
  RESUMEN PASO 2
══════════════════════════════════════════════════════════════

  Modelo seleccionado:      XGBoost
  Features utilizadas:      32
  Jugadores entrenados:     17,035

  Métricas OOF (escala log):
    RMSE:   0.2817
    R²:     0.9711
    MAE:    0.1865

  Métricas en escala original:
    RMSE:   2.12M€
    MAE:    0.41M€

  Outputs generados:
    ranking_promesas.csv
    scouting_dataset_v2_con_predicciones.csv
    08 plots PNG en outputs/paso2/
    Experimentos en mlruns/ (ver con: mlflow ui)


## Conclusiones del Paso 2 y próximos pasos

### Lo que tenemos listo
- Modelo predictivo entrenado y validado con 5-Fold CV
- Comparación cuantitativa LightGBM vs XGBoost
- Análisis de residuos: sabemos dónde el modelo es preciso y dónde no
- SHAP values: explicabilidad de cada predicción individual
- Ranking completo de 31,405 jugadores con valor predicho
- Todos los experimentos registrados en MLflow

### ⚠️ Limitaciones del modelo actual
| Limitación | Impacto | Plan Paso 3 |
|---|---|---|
| Predice valor *actual*, no futuro | El reclutador necesita proyección temporal | Montecarlo para t+3 y t+5 años |
| Subestima jugadores < 20 años con pocos stats | Los más jóvenes tienen historiales cortos | Ponderar potencial por edad en Montecarlo |
| No captura rendimiento reciente | Usa toda la carrera por igual | Feature de "últimas 2 temporadas" en v2 |
| Ligas de academias y categorías menores con ruido | Distorsiona stats de jugadores juveniles | Filtro por nivel mínimo de competición |